Nella lezione precedente abbiamo visto due estremi: il cifrario a sostituzione è insicuro, mentre il one-time pad offre segretezza perfetta ma richiede una chiave davvero casuale e lunga quanto il messaggio, nonché utilizzabile una sola volta per garantire integrità. Gestire chiavi di questo tipo per tutte le comunicazioni è **impraticabile**, occorrono cifrari utilizzabili nel mondo reale.

Si presentano quindi due famiglie di cifrari:
1. **Stream Ciphers**: sono cifrari che generano un **keystream** da combinare con il messaggio per ottenere il testo cifrato. Esempi di questa famiglia sono RC4, Salsa20 e ChaCha20. 
2. **Block Ciphers**: cifrari che operano su blocchi di dati di dimensione fissa (ad esempio 128 bit) e utilizzano una chiave per trasformare il blocco di testo in un blocco di testo cifrato. Esempi di questa famiglia sono AES, DES e 3DES.

AES è un cifrario a blocchi, ma alcune modalità d'uso come AES-CTR gli permettono di produrre un keystream e di funzionare sotto questo aspetto come uno stream cipher. 

## Stream Ciphers
Uno stream cipher parte da una **chiave segreta** $K$ (corta, ad esempio di 128 o 256 bit) e a partire da essa genera una sequenza pseudocasuale $S$, detta **keystream**, di lunghezza pari a quanto necessario per cifrare il messaggio.

Quindi **chiave** e **keystream** sono **due oggetti diversi**: la chiave è il segreto relativamente corto condiviso tra mittente e destinatario, mentre $S$ è la lunga sequenza generata a partire da $K$, usata concretamente nello XOR con il messaggio.  
Inoltre, mentre nel one-time pad la sequenza usata nello XOR è veramente causale, nello stream cipher è **pseudocasuale**: un algoritmo la produce deterministicamente a partire da un segreto corto. Questa differenza ci fa passare dalla **segretezza perfetta** alla **segretezza computazionale**: *la sicurezza dipende dalla difficoltà pratica di distinguere o prevedere il keystream e di trovare la chiave*.

Per il corso non entreremo in dettaglio sul funzionamento interno del generatore, l'obiettivo è capire come usare correttamente un cifrario e non come progettarlo.

<img src="imgs/stream_cipher.png" width="400">

Quindi in pratica anzitutto la chiave viene convertita nel keystream tramite prng, poi encode di M con lo XOR ottenendo C, infine il destinatario si ricava anch'esso il keystream e decifra il messaggio C con lo stesso XOR.

La cifratura e la decifratura usano la stessa proprietà dello XOR vista per il one-time pad:
$$C = M \oplus S \quad M = C \oplus S$$
Se all'interno dello stesso messaggio compare più volte "CIAO", ciascuna occorrenza "incontra una porzione diversa" del keystream e per questo le ripetizioni del plaintext non producono necessariamente blocchi di ciphertext uguali. Inoltre in generale anche cambiando un solo bit del plaintext, il ciphertext cambia in maniera imprevedibile.

Resta tuttavia il problema per cui **se si avvia ogni volta il generatore con la stessa chiave, ottengo sempre lo stesso keystream e quindi lo stesso testo cifrato.** Ciò si ricollega alla sicurezza semantica: il cifrario così costruito non è IND-CPA (l'avversario può distinguere quale messaggio è stato cifrato semplicemente cifrando a priori uno dei due messaggi). 

### Initialization Vector (IV)
Come fare quindi a far sì di utilizzare una chiave condivisa, che può anche rimanere installata nel dispositivo molto a lungo, senza ripetere il keystream per ogni messaggio? La soluzione è introdurre un **Initialization Vector (IV)**, che cambia tra cifrature diverse.

Si rappresenta concettualmente come segue:
$$S_i = PRNG(K || IV_i) \quad C_i = M_i \oplus S_i$$
In pratica cioè lo stesso segreto viene concatenato ogni volta con un IV diverso, producendo così un keystream diverso per ogni messaggio. Il destinatario ha già $K$, ma deve sapere quale IV è stato usato --> l'IV viene **inviato in chiaro assieme al ciphertext**. Ricevuto $(IV_i, C_i)$, il destinatario rigenera $S_i$ e calcola $M_i = C_i \oplus S_i$.

Quindi è importante capire che l'IV **non è segreto** e che **lo stesso IV non deve ripetersi con la stessa chiave**. Se si ripete la coppia (K, IV) infatti, si ripete anche il keystream e viene meno la sicurezza semantica.  
Per i nostri scopi si assume un IV generato casualmente ad ogni messaggio, tuttavia questo dipende dallo schema specifico, l'importante è la non ripetizione con la stessa chiave.

### WEP
La tesi centrale della lezione è che **anche scegliendo un buon algoritmo crittografico, si può costruire un sistema insicuro se lo si usa male nel protocollo**.

WEP era un sistema di protezione delle prime reti Wi-Fi. Doveva fornire autenticazione (i.e. il fatto che l'utente sia chi dice di essere), integrità (i.e. il fatto che i dati non vengano modificati durante la trasmissione) e confidenzialità (i.e. il fatto che i dati non siano leggibili da chi non è autorizzato), ma tutti i tre obiettivi fallirono. 

WEP usava RC4 come stream cipher. Ipotizziamo RC4 come modello semplificato:
$$S = RC4(K,IV) \quad C = M \oplus S$$
RC4 ha debolezze intrinseche di cui parleremo poi, ma per ora immaginiamolo come cifrario perfetto per dimostrare come il problema non sia l'algoritmo ma WEP in sé nella gestione degli IV.

In una trasmissione a pacchetti, un frame può andare perso. Se mittente e destinatario avanzassero lungo un unico keystream continuo, allora la perdita di un singolo frame potrebbe far perdere loro la sincronizzazione (ad ogni frame infatti l'IV cambia e quindi anche il keystream!).  
Per risolvere il problema, WEP **usa un IV per ogni frame**: ciascun frame può essere decifrato indipendentemente (il destinatario prende l'IV ricevuto, lo combina con la chiave condivisa e ricostruisce quindi il keystream necessario per decifrare il pacchetto).  
Si noti come se un IV torna ad essere usato con la stessa chiave, indicando con $S$ il keystream associato alla coppia (K, IV), l'osservatore può fare lo xor tra i due ciphertext e se conosce uno dei due plaintext può ricavare l'altro:
$$C_1 = M_1 \oplus S \quad C_2 = M_2 \oplus S \quad C_1 \oplus C_2 = M_1 \oplus M_2$$

Assumendo però che gli IV siano sempre differenti per ogni messaggio, vediamo di capire gli errori nella progettazione di WEP. 
1. **L'IV è di soli 24 bit (3 byte)**. Il motivo è che dovendo trasmettere l'IV con il pacchetto, è conveniente che l'IV occupi poco spazio.  
In questo modo gli IV possibili sono $2^{24} \approx 16.7$ milioni. Questo sembra un numero grande, ma in una rete Wi-Fi si generano frame continuamente. Se gli IV sono stati usati in sequenza, dopo aver attraversato tutti i valori si ricomincia. Con frame da 1500 byte e una banda di 7 Mbit/s in meno di 8 ore gli IV si ripetono sotto traffico continuo.  
Se invece gli IV sono generati casualmente, per il paradosso del compleanno è sufficiente aspettare un numero dell'ordine di $\sqrt{2^{24}} \approx 4096$ per avere una collisione (dati n frame, ho $\frac{n(n-1)}{2}$ coppie possibili di questi. $N = 2^{24}$ possibili IV, e per ogni coppia fisso uno e l'altro ah probabilità $\frac{1}{2^{24}}$ di coincidere. Quindi una collisione diventa plausibile quando il numero di coppie è comparabile con $N$, ossia $\frac{n^2}{2} \approx N$ ergo $n \approx \sqrt{N}$).
1. **La scelta degli IV è lasciata all'implementazione**. Dopo un riavvio del dispositivo, l'IV può ripartire da 0 ripetendo subito quindi i keystream, senza neppure aspettare che lo spazio degli IV venga esaurito. 

#### **Rottura della confidenzialità (l'attacco del dizionario)**
Supponiamo che l'attaccante conosca il plaintext $M$ di un frame, veda l'IV e intercetti il ciphertext $C$. Allora può calcolare il keystream: 
$$C = M \oplus S \implies S = C \oplus M$$
In particolare potrebbe memorizzare una voce del tipo $(IV, S)$, sarebbe una sorta di dizionario indicizzato dall'IV. Se in futuro intercettasse un frame **con stesso IV e stessa chiave**, può prendere il keystream salvato e decifrare il messaggio:
$$M' = C' \oplus S$$
Attenzione al fatto che l'attaccante non ha scoperto la chiave segreta $K$, ma il keystream relativo a uno specifico IV. Questo gli basta a decifrare tutti i frame per cui l'IV coincide.

**La confidenzialità di WEP è quindi compromessa.**

Come risalire a plaintext noti? Un'idea per l'attaccante potrebbe essere quella di inviare lui stesso dei frame con plaintext scelto da lui arbitrariamente.  
Altrimenti un caso ancora più diretto sta nel fatto che la procedura di autenticazione WEP fornisce essa stessa plaintext noti, come vedremo.

Chiaro inoltre che modificare la lunghezza della chiave non risolve il problema: l'attaccante non ha bisogno di conoscere la chiave, ma solo il keystream relativo a un IV specifico.

Inoltre ricorda che qua si ipotizza RC4 cifrario perfetto. In realtà non lo è affatto, alcune scelte di IV producono regolarità statistiche che, osservando molti pacchetti, possono aiutare a ricavare informazioni sulla chiave (vedi attacchi di Fluhrer, Mantin e Shamir). Non si sviluppa l'algoritmo di attacco, ma ricorda che in Wi-Fi è utile conoscere i primi byte del messaggio in quanto l'incapsulamento LLC/SNAP rende prevedibile il contenuto del primo byte, tutte informazioni che aiutano a ricavare il keystream e quindi a decifrare i messaggi ma intrinseche nella debolezza di RC4 che ora stiamo ignorando.

#### **Rottura dell'autenticazione**
Con **autenticazione** si intende **verificare che chi chiede accesso conosca il segreto richiesto**.

WEP richiede al dispositivo di dimostrare di conoscere una chiave condivisa. Chiaramente non può farsela inviare in chiaro perché chiunque ascolti la apprenderebbe. La soluzione scelta da WEP è quindi un protocollo **challenge-response**:
1. L'access point invia una sfida casuale, detta challenge
2. Il dispositivo la cifra con la chiave condivisa e la invia indietro
3. L'access point decifra la risposta e verifica che coincida con la sfida inviata

Inviando di volta in volta una challenge diversa si evita che l'attancante possa semplicemente intercettare una risposta precedente e riutilizzarla per autenticarsi. Tuttavia i problemi stanno nei dettagli della specifica cifratura usata da WEP.  

Il primo dettaglio problematico è che WEP **usa la stessa chiave per autenticazione e cifratura dei dati**. In generale una regola progettuale per un qualsiasi protocollo di cifratura è che **non si deve mai usare la stessa chiave per scopi diversi**.  
La challenge $Q$ viene inviata in chiaro, nota quindi a chiunque ascolti. La risposta WEP contiene l'IV scelto IN CHIARO e il ciphertext:
$$C = Q \oplus RC4(K, IV)$$
L'attaccante ha quindi tutti i valori necessari per calcolare il keystream $S = RC4(K, IV) = Q \oplus C$, ottenendo così una nuova voce per il dizionario $(IV, S)$.  
Ogni autenticazione osservata offre cioè materiale potenzialmente utile per attaccare la confidenzialità dei dati cifrati con la stessa chiave.  
In questo senso è da considerarsi inoltre un AP malevolo che induce un dispositivo a rispondere con più challenge per raccogliere molte coppie IV-keystream.

In generale però vediamo di capire **perché l'autenticazione WEP è effettivamente rotta**.  
Supponiamo che l'attaccante abbia ascoltato una sola autenticazione legittima:
- challenge nota $Q$,
- IV pubblico $v$,
- risposta intercettata $C = Q \oplus S_v$, dove $S_v = RC4(K, v)$ è il keystream relativo a quell'IV.

Allora come detto l'attaccante può calcolare S_v = Q \oplus C. Quando prova ad autenticarsi lui, l'AP gli invia una **nuova** challenge $Q'$, allora l'attaccante può rispondere con:
$$v, \quad C' = Q' \oplus S_v$$
L'AP ricostruisce lo stesso keystream $S_v$ a partire dalla propria chiave, decifra $C'$ e ritrova $Q'$, autenticando l'attaccante **nonostante egli non conosca $K$.**

**Il dettaglio decisivo qui sta nel fatto che nella risposta WEP è colui che si autentica (i.e. chi deve cifrare) a scegliere l'IV da usare.** In questo modo all'attaccante basta scegliere proprio l'IV $v$ di cui conosce il keystream, che può riutilizzare per cifrare la nuova challenge.

L'integrità si vede nella prossima lezione.